# IndiaStay Hotels — Data Cleaning (Step 1)

This notebook cleans the raw hotel booking dataset, **one small step at a time**.

Every messy dataset usually has the same kinds of problems:
- duplicate rows
- inconsistent spelling (e.g. "Bangalore" vs "Bengaluru")
- missing values
- wrong data types (dates stored as text)
- impossible/incorrect values (like a negative price)

We will fix each problem **one at a time**, and after each fix we will
print something to *prove* it worked. That way you can see exactly
what changed and why.

## Step 0: Import the tools we need

- `pandas` → for working with tables of data (like Excel, but in code)
- `numpy` → for handling numbers and missing values

In [ ]:
import pandas as pd
import numpy as np

## Step 1: Load the raw data

We read the CSV file into a **DataFrame** — pandas' name for a table.

In [ ]:
df = pd.read_csv("indiastay_hotel_bookings_raw.csv")

print("Shape of data (rows, columns):", df.shape)
df.head()

## Step 2: Get a first look at the data

Before cleaning anything, always look at:
1. What columns exist and what data type they are (`.info()`)
2. How many values are missing in each column (`.isnull().sum()`)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## Step 3: Remove fully duplicate rows

Sometimes the exact same booking row appears twice by mistake
(e.g. from a system glitch). We use `.drop_duplicates()` to remove
rows that are 100% identical across every column.

In [ ]:
before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"Removed {before - after} fully duplicate rows")
print("New shape:", df.shape)

## Step 4: Remove duplicate Booking IDs

Each `Booking_ID` should be unique — it's like a receipt number.
If the same ID appears more than once, that's a data error, so we
keep only the **first** time we see it and drop the rest.

In [ ]:
duplicate_id_count = df['Booking_ID'].duplicated().sum()
df = df.drop_duplicates(subset='Booking_ID', keep='first')

print(f"Removed {duplicate_id_count} rows with a duplicate Booking_ID")
print("New shape:", df.shape)

## Step 5: Fix inconsistent City names

Look closely and you'll find the same city written in different ways:
`Bangalore`, `bengaluru`, `BENGALURU` are all the same city.

We build a small **dictionary** (a lookup table) that maps every messy
version to one clean, correct name.

In [ ]:
print("Examples of messy city spellings currently in the data:")
print(df['City'].dropna().unique()[:20])

In [ ]:
city_map = {
    "ahmedabad": "Ahmedabad", "ahmadabad": "Ahmedabad",
    "bangalore": "Bengaluru", "bengaluru": "Bengaluru",
    "mumbai": "Mumbai", "bombay": "Mumbai",
    "delhi": "Delhi", "new delhi": "Delhi",
    "cochin": "Kochi", "kochi": "Kochi",
    "baroda": "Vadodara", "vadodara": "Vadodara",
}

def clean_city(value):
    if pd.isna(value):
        return np.nan
    key = str(value).strip().lower()          # remove spaces, make lowercase
    return city_map.get(key, str(value).strip().title())  # look up, else Title Case

df['City'] = df['City'].apply(clean_city)

print("City names after cleaning:")
print(df['City'].dropna().unique())

## Step 6: Fix inconsistent Room Type names

Same idea as Step 5, but for `Room_Type` (e.g. `DLX`, `Delux`, `deluxe`
should all just be `Deluxe`).

In [ ]:
room_map = {
    "std": "Standard", "std.": "Standard", "standard": "Standard",
    "dlx": "Deluxe", "delux": "Deluxe", "deluxe": "Deluxe",
    "exec": "Executive", "executive": "Executive",
    "ste": "Suite", "suite": "Suite",
    "prem": "Premium", "premium": "Premium",
}

def clean_room_type(value):
    if pd.isna(value):
        return np.nan
    key = str(value).strip().lower()
    return room_map.get(key, str(value).strip().title())

df['Room_Type'] = df['Room_Type'].apply(clean_room_type)

print("Room types after cleaning:")
print(df['Room_Type'].unique())

## Step 7: Fill in missing State using City

If we know the `City`, we can usually figure out the `State` from
other rows that have the same city filled in correctly. This is
called **imputation** — filling missing values using other data we
already have, instead of just guessing randomly.

In [ ]:
# Build a City -> most common State lookup, using only rows where State is known
city_to_state = (
    df.dropna(subset=['City', 'State'])
      .groupby('City')['State']
      .agg(lambda s: s.mode().iloc[0])   # most frequent state for that city
      .to_dict()
)

missing_before = df['State'].isna().sum()

mask = df['State'].isna() & df['City'].notna()
df.loc[mask, 'State'] = df.loc[mask, 'City'].map(city_to_state)

missing_after = df['State'].isna().sum()
print(f"Missing State values: {missing_before} -> {missing_after}")

## Step 8: Convert date columns from text to real dates

Right now `Booking_Date`, `Check_In`, and `Check_Out` are just text,
and they're written in *different formats* in different rows
(e.g. `2025-03-19`, `19/03/2025`, `19 Mar 2025`).

`pd.to_datetime(..., format='mixed')` reads all these different text
formats and converts them into one real, consistent date type that
pandas can do math with (like subtracting two dates).

In [ ]:
for col in ['Booking_Date', 'Check_In', 'Check_Out']:
    df[col] = pd.to_datetime(df[col], errors='coerce', format='mixed')

df[['Booking_Date', 'Check_In', 'Check_Out']].head()

## Step 9: Fill missing Nights

`Nights` tells us how many nights the guest stayed. If it's missing
but we know `Check_In` and `Check_Out`, we can simply calculate it:

`Nights = Check_Out - Check_In`

If we can't calculate it either (both dates missing), we fill the gap
with the **median** (the middle/typical value) so we don't have to
throw the row away.

In [ ]:
calculated_nights = (df['Check_Out'] - df['Check_In']).dt.days
df['Nights'] = df['Nights'].fillna(calculated_nights)
df['Nights'] = df['Nights'].fillna(df['Nights'].median())
df['Nights'] = df['Nights'].clip(lower=1)   # a stay can't be 0 or negative nights

print("Missing Nights remaining:", df['Nights'].isna().sum())

## Step 10: Fix Check_Out dates that don't match Nights

Here's a tricky real-world problem: a date like `12-02-2025` could
mean **Dec 2** or **Feb 12** depending on the format — that's an
*ambiguous* date. This can cause `Check_Out` to be read incorrectly.

We catch this by checking: does `Check_Out - Check_In` actually equal
`Nights`? If not, something was misread — so we trust the `Nights`
number (a plain number can't be ambiguous the way a date can) and
recalculate `Check_Out` from it.

In [ ]:
implied_gap = (df['Check_Out'] - df['Check_In']).dt.days
mismatched = implied_gap.isna() | (implied_gap != df['Nights'])

print(f"Found {mismatched.sum()} rows where Check_Out didn't match Nights")

df.loc[mismatched, 'Check_Out'] = (
    df.loc[mismatched, 'Check_In'] + pd.to_timedelta(df.loc[mismatched, 'Nights'], unit='D')
)

## Step 11: Fix the Room Rate column

Two problems here:
1. A few rates are **negative** (a data-entry mistake) → fix with `.abs()`
   which just removes the minus sign.
2. Some rates are **missing** → fill with the median rate for that
   same `Room_Type` (a Suite's missing price should look like other
   Suite prices, not the overall average of everything).

In [ ]:
df['Room_Rate'] = df['Room_Rate'].abs()

rate_by_room_type = df.groupby('Room_Type')['Room_Rate'].median()
print("Typical (median) rate per room type:")
print(rate_by_room_type)

In [ ]:
def fill_rate(row):
    if pd.isna(row['Room_Rate']) and pd.notna(row['Room_Type']):
        return rate_by_room_type[row['Room_Type']]
    return row['Room_Rate']

df['Room_Rate'] = df.apply(fill_rate, axis=1)
print("Missing Room_Rate remaining:", df['Room_Rate'].isna().sum())

## Step 12: Fill remaining simple missing values

- `Discount_Percent` missing → assume **0%** (no discount was applied)
- `Booking_Channel`, `Payment_Method`, `City`, `State` missing →
  label as `"Unknown"` instead of guessing
- `Guests` missing → fill with the median number of guests

In [ ]:
df['Discount_Percent'] = df['Discount_Percent'].fillna(0)

for col in ['Booking_Channel', 'Payment_Method', 'City', 'State']:
    df[col] = df[col].fillna("Unknown")

df['Guests'] = df['Guests'].fillna(df['Guests'].median())

df.isnull().sum()

## Step 13: Fix impossible Rating values

Ratings should be between 0 and 5. A few rows have values like `5.4`
or `5.8`, which shouldn't be possible. We **cap** them at 5.0 using
`.clip(upper=5.0)` — any value higher than 5 becomes exactly 5.

In [ ]:
print("Highest rating before fix:", df['Rating'].max())
df['Rating'] = df['Rating'].clip(upper=5.0)
print("Highest rating after fix:", df['Rating'].max())

## Step 14: Recalculate Total_Amount

Now that `Room_Rate`, `Nights`, and `Discount_Percent` are all clean,
we recalculate `Total_Amount` from scratch so it's always consistent
with the other columns:

`Total_Amount = Room_Rate x Nights x (1 - Discount / 100)`

In [ ]:
df['Total_Amount'] = (
    df['Room_Rate'] * df['Nights'] * (1 - df['Discount_Percent'] / 100)
).round(2)

df[['Room_Rate', 'Nights', 'Discount_Percent', 'Total_Amount']].head()

## Step 15: Add a few helpful new columns

These aren't "cleaning" exactly — they're small extra columns
calculated from what we already have, which will make the next
steps (EDA, SQL, Power BI) much easier:

- `Lead_Time_Days` → how many days in advance the booking was made
- `Booking_Month` / `Booking_Year` → for looking at seasonal trends
- `Is_Cancelled` → a simple 1/0 version of Cancellation, useful for
  calculations later (e.g. cancellation rate)

In [ ]:
df['Lead_Time_Days'] = (df['Check_In'] - df['Booking_Date']).dt.days
df['Booking_Month'] = df['Booking_Date'].dt.month_name()
df['Booking_Year'] = df['Booking_Date'].dt.year
df['Is_Cancelled'] = (df['Cancellation'] == 'Yes').astype(int)

df[['Lead_Time_Days', 'Booking_Month', 'Booking_Year', 'Is_Cancelled']].head()

## Step 16: Final check before saving

Let's confirm the shape of the data and see which columns (if any)
still have missing values — a few are *expected* to still be missing
and that's okay:
- `Rating` is only filled in for guests who have checked out
- A handful of dates may still be missing if both the raw date *and*
  a way to calculate it were missing

In [ ]:
print("Final shape:", df.shape)
df.isnull().sum()[df.isnull().sum() > 0]

In [ ]:
df.head(10)

## Step 17: Save the cleaned dataset

Finally, we save our clean table as a new CSV file so we can use it
in the next steps (EDA, SQL, Power BI) without redoing any of this
work.

In [ ]:
df.to_csv("indiastay_hotel_bookings_clean.csv", index=False)
print("Saved cleaned file: indiastay_hotel_bookings_clean.csv")